In [ ]:
# Install playwright, its system dependencies, and the chromium browser
!pip install playwright
!playwright install-deps chromium
!playwright install chromium

In [6]:
import asyncio
import os
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

async def extract_blog_data():
    # File paths
    input_file = 'blog_links.txt'
    output_file = 'blog_data_extracted.txt'

    # Check if input file exists
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found. Please upload it to your Colab environment.")
        return

    # Read all links from the file
    with open(input_file, 'r', encoding='utf-8') as f:
        links = [line.strip() for line in f if line.strip().startswith('http')]

    # --- ADDED FOR TESTING: Limit to just the first 2 links ---
    # links = links[:10]

    print(f"Total links found: {len(links)}. Starting full extraction...")

    async with async_playwright() as p:
        # Launch browser with robust settings for server/notebook environments
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-setuid-sandbox"])
        context = await browser.new_context()
        page = await context.new_page()

        # Initialize output file (overwrite with header)
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("BLOG DATA EXTRACTION - FULL COLLECTION\n")

        success_count = 0

        for index, url in enumerate(links, 1):
            # Extract a slug from the URL for logging purposes
            url_parts = [part for part in url.split('/') if part]
            blog_slug = url_parts[-1] if url_parts else f"blog_link_{index}"

            print(f"[{index}/{len(links)}] Processing: {blog_slug}...")

            try:
                # Navigate to the blog page
                await page.goto(url, timeout=60000, wait_until="domcontentloaded")

                # Small delay to let dynamic content/images load
                await page.wait_for_timeout(2000)

                # Extract HTML and parse
                html_content = await page.content()
                soup = BeautifulSoup(html_content, 'html.parser')

                extracted_text = ""

                # Target the title element
                # <h1 class="entry-title td-module-title">
                title_el = soup.find('h1', class_='entry-title td-module-title')
                title_text = title_el.get_text(strip=True) if title_el else f"Title for {blog_slug.replace('-', ' ').title()}"

                # Build formatted header for the blog title with ===
                header_line = "=" * len(title_text)
                formatted_title = f"{header_line}\n{title_text}\n{header_line}\n"

                extracted_text += formatted_title + "\n"

                # Target the main content container
                # <div class="td-post-content tagdiv-type">
                content_container = soup.find('div', class_='td-post-content tagdiv-type')

                if content_container:
                    # Extract the text, separating paragraphs by newlines
                    body_text = content_container.get_text(separator='\n', strip=True)

                    extracted_text += "--- CONTENT ---\n"
                    extracted_text += body_text + "\n\n"

                    # Write results to file ONLY if the content container was found
                    if extracted_text.strip():
                        with open(output_file, 'a', encoding='utf-8') as f:
                            # Link separators using ===
                            f.write(f"\n\n{'='*60}\nURL: {url}\n{'='*60}\n\n")
                            f.write(extracted_text)
                        print(f"  -> SUCCESS: Saved.")
                        success_count += 1
                else:
                    print(f"  -> SKIPPED: No main content found matching class 'td-post-content tagdiv-type'")

            except Exception as e:
                print(f"  -> ERROR: {e}")

        # Final cleanup
        try:
            await browser.close()
        except:
            pass

        print(f"\nExtraction complete! Successfully saved {success_count} out of {len(links)} links. Results are in {output_file}")

# Execution for Jupyter/Colab environment
await extract_blog_data()

Total links found: 2408. Starting full extraction...
[1/2408] Processing: 100179-units-for-sale-in-june-north-coast-with-a-5-dp...
  -> SKIPPED: No main content found matching class 'td-post-content tagdiv-type'
[2/2408] Processing: 100180-best-summer-houses-for-sale-in-june-ras-el-hikma...
  -> SKIPPED: No main content found matching class 'td-post-content tagdiv-type'
[3/2408] Processing: 100189-townhouses-in-swan-lake-west-with-a-5-down-payment...
  -> SKIPPED: No main content found matching class 'td-post-content tagdiv-type'
[4/2408] Processing: 100194-june-ras-el-hikma-apartments-delivery-in-2026...
  -> SKIPPED: No main content found matching class 'td-post-content tagdiv-type'
[5/2408] Processing: 100222-3-reasons-to-choose-koun-village-for-your-beach-unit...
  -> SUCCESS: Saved.
[6/2408] Processing: 100258-naia-west-a-perfectly-integrated-luxury...
  -> SUCCESS: Saved.
[7/2408] Processing: 100289-hassan-allam-october-prices-revealed...
  -> SKIPPED: No main content found match